In [1]:
!pip install torchmetrics tqdm roboflow wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 61.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 107.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 95.6 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalle

In [2]:
import os
import torch
import cv2
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
import wandb
from kaggle_secrets import UserSecretsClient

In [3]:
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("ROBO_FLOW_API")

rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("obeds-workspace-m260m").project("benchmarking-models-road-anom")
version = project.version(2)
dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Benchmarking-Models---Road-Anom.-2 in yolov8:: 100%|██████████| 8893/8893 [00:01<00:00, 7195.71it/s] 


In [4]:
# 1. Authenticate W&B
user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY_DLI"))
run = wandb.init(project="road-anomaly-benchmark", name="faster-rcnn-baseline")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: obedhonoureje (obedhonoureje-federal-university-of-technology-minna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260602_173010-ycrx8ky3
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run faster-rcnn-baseline
wandb: ⭐️ View project at https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark
wandb: 🚀 View run at https://wandb.ai/obedhonoureje-federal-university-of-

In [5]:
# 2. Build the Raw Dataset Class
class YoloToFasterRCNNDataset(Dataset):
    def __init__(self, img_dir, label_dir):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.imgs = [img for img in os.listdir(img_dir) if img.endswith('.jpg') or img.endswith('.png')]

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        
        # Load image and convert to PyTorch Tensor (C, H, W)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        height, width, _ = img.shape
        img_tensor = torch.as_tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0

        label_path = os.path.join(self.label_dir, img_name.rsplit('.', 1)[0] + '.txt')
        boxes = []
        labels = []

        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    class_id, x_center, y_center, w, h = map(float, line.strip().split())
                    
                    # MATH: Convert YOLO (Center, W, H) to Faster R-CNN (X_min, Y_min, X_max, Y_max)
                    xmin = (x_center - w / 2) * width
                    xmax = (x_center + w / 2) * width
                    ymin = (y_center - h / 2) * height
                    ymax = (y_center + h / 2) * height
                    
                    boxes.append([xmin, ymin, xmax, ymax])
                    # Faster R-CNN requires class 0 to be the 'background'. 
                    # We shift our classes (Pothole=0, Speedbump=1) up by 1.
                    labels.append(int(class_id) + 1) 

        # If no objects, provide empty tensors to prevent crashes
        if len(boxes) == 0:
            boxes = torch.empty((0, 4), dtype=torch.float32)
            labels = torch.empty((0,), dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)

        target = {"boxes": boxes, "labels": labels}
        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

In [6]:
# 3. Setup Data Loaders (Make sure the path matches your Roboflow download!)
# e.g., dataset_path = "/kaggle/working/benchmarking-models-road-anom-1"
dataset_path = dataset.location 

train_dataset = YoloToFasterRCNNDataset(f"{dataset_path}/train/images", f"{dataset_path}/train/labels")
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn)

val_dataset = YoloToFasterRCNNDataset(f"{dataset_path}/valid/images", f"{dataset_path}/valid/labels")
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, collate_fn=collate_fn)

In [7]:
# 4. Initialize the Faster R-CNN Architecture
print("\n--- Initializing Faster R-CNN ResNet-50 ---")
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

# Replace the pre-trained head with a new one for our specific classes
# 3 classes: 1 (Pothole), 2 (Speedbump), + 1 (Background) = 3
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes=3)

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)


--- Initializing Faster R-CNN ResNet-50 ---
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth" to /root/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_coco-258fb6c6.pth


100%|██████████| 160M/160M [00:00<00:00, 175MB/s]


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [8]:
# 5. The Training Loop (With tqdm Progress Bar & Validation)
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)
num_epochs = 20 

print("\n--- Starting Raw PyTorch Training ---")
for epoch in range(num_epochs):
    # ---------------------------------------------------------
    # PHASE 1: TRAINING
    # ---------------------------------------------------------
    model.train() # Set model to training mode
    epoch_loss = 0
    
    # Wrap train_loader in tqdm for the progress bar
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=True)
    
    for images, targets in train_loop:
        images = list(image.to(device) for image in images)
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass calculates all RPN and Box regression losses
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
        train_loop.set_postfix(loss=losses.item())
        
    avg_loss = epoch_loss / len(train_loader)
    
    # ---------------------------------------------------------
    # PHASE 2: VALIDATION & METRICS
    # ---------------------------------------------------------
    print(f"\nEvaluating Epoch {epoch+1} on Validation Set...")
    model.eval() # Freezes weights, turns off dropout/batchnorm
    
    metric = MeanAveragePrecision(iou_type="bbox")
    
    with torch.no_grad(): # Don't track gradients during validation
        val_loop = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Valid]", leave=True)
        for val_images, val_targets in val_loop:
            val_images = list(img.to(device) for img in val_images)
            val_targets = [{k: v.to(device) for k, v in t.items()} for t in val_targets]
            
            predictions = model(val_images)
            metric.update(predictions, val_targets)
            
    # Compute the final mAP scores
    metrics_dict = metric.compute()
    map50 = metrics_dict['map_50'].item()
    map50_95 = metrics_dict['map'].item()
    
    print(f"Epoch {epoch+1} Summary -> Train Loss: {avg_loss:.4f} | mAP@50: {map50:.4f} | mAP@50-95: {map50_95:.4f}\n")
    
    # Log EVERYTHING to WandB once per epoch
    wandb.log({
        "epoch": epoch + 1,
        "train/loss": avg_loss, 
        "val/mAP50": map50, 
        "val/mAP50-95": map50_95
    })

# Save the final raw PyTorch weights
torch.save(model.state_dict(), "/kaggle/working/faster_rcnn_best.pth")
wandb.finish()
print("✓ Faster R-CNN Pipeline Complete!")


--- Starting Raw PyTorch Training ---


Epoch 1/20 [Train]:   0%|          | 0/778 [00:00<?, ?it/s]


Evaluating Epoch 1 on Validation Set...


Epoch 1/20 [Valid]:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch 1 Summary -> Train Loss: 0.3117 | mAP@50: 0.6234 | mAP@50-95: 0.2883



...


Evaluating Epoch 20 on Validation Set...


Epoch 20/20 [Valid]:   0%|          | 0/167 [00:00<?, ?it/s]

Epoch 20 Summary -> Train Loss: 0.0822 | mAP@50: 0.7806 | mAP@50-95: 0.4419



wandb: updating run metadata
wandb: 
wandb: Run history:
wandb:        epoch ▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
wandb:   train/loss █▆▅▄▄▃▃▃▂▂▂▂▂▁▁▁▁▁▁▁
wandb:    val/mAP50 ▁▄▅▆▆▇▇▇▇█▇▇█▇██▇▇▇▇
wandb: val/mAP50-95 ▁▄▄▆▆▇▇▇▇▇▇▇█████▇▇▇
wandb: 
wandb: Run summary:
wandb:        epoch 20
wandb:   train/loss 0.08215
wandb:    val/mAP50 0.78062
wandb: val/mAP50-95 0.44195
wandb: 
wandb: 🚀 View run faster-rcnn-baseline at: https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark/runs/ycrx8ky3
wandb: ⭐️ View project at: https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260602_173010-ycrx8ky3/logs


✓ Faster R-CNN Pipeline Complete!
